# When kernels go flat: exponential concentration
**Quantum Machine Learning · Week 2 (Quantum kernels) · Day 4**

Lesson page: https://sara-aminpour.com/tutorials/qml/w2d4.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

Add qubits to a quantum kernel and something alarming happens: the kernel values of different points
shrink towards a tiny constant, about $2^{-n}$. This is **exponential concentration**, and it can make a
quantum kernel useless, however clever its feature map.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

def zz_map(x, reps=2):
    n = len(x)
    qc = QuantumCircuit(n)
    for _ in range(reps):
        qc.h(range(n))
        for q in range(n):
            qc.p(2 * x[q], q)
        for q in range(n - 1):
            qc.cx(q, q + 1)
            qc.p(2 * (np.pi - x[q]) * (np.pi - x[q + 1]), q + 1)
            qc.cx(q, q + 1)
    return qc

def kernel_matrix(U, s):
    S = np.array([Statevector(zz_map(s * u)).data for u in U])
    return np.abs(S.conj() @ S.T) ** 2

rng = np.random.default_rng(0)
ns = [2, 4, 6, 8, 10, 12]
means, spreads = [], []
for n in ns:
    K = kernel_matrix(rng.uniform(0, 1, (30, n)), np.pi)
    off_diagonal = K[~np.eye(30, dtype=bool)]
    means.append(off_diagonal.mean())
    spreads.append(off_diagonal.std())
    print(f"n = {n:2d}: average kernel value {means[-1]:.5f}, spread {spreads[-1]:.5f}, 2^-n = {2.0 ** -n:.5f}")

plt.semilogy(ns, means, "o-", color="#7b5cfa", label="average kernel value")
plt.semilogy(ns, spreads, "s-", color="#ff7a59", label="spread")
plt.semilogy(ns, 2.0 ** -np.array(ns), "--", color="#8a85a3", label=r"$2^{-n}$")
plt.xlabel("qubits n")
plt.legend()
plt.show()

## The bandwidth to the rescue
Multiply the features by a small bandwidth $s$: the encoded states stay close together, and the kernel
values keep their spread.

In [ ]:
U = rng.uniform(0, 1, (30, 10))
for s in [0.1, 0.25, 0.5, 1.0, np.pi]:
    off_diagonal = kernel_matrix(U, s)[~np.eye(30, dtype=bool)]
    print(f"10 qubits, s = {s:4.2f}: average {off_diagonal.mean():.4f}, spread {off_diagonal.std():.4f}")

## What it does to learning
Handwritten 3s versus 8s with 10 PCA features on 10 qubits. With $s = \pi$ the kernel concentrates; a small
bandwidth keeps it useful. Here the kernels are still exact (no shot noise yet).

In [ ]:
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

digits = load_digits()
mask = np.isin(digits.target, [3, 8])
X, y = digits.data[mask], (digits.target[mask] == 8).astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
pca = PCA(n_components=10).fit(X_train)
scaler = MinMaxScaler((0, 1)).fit(pca.transform(X_train))
U_train, U_test = scaler.transform(pca.transform(X_train)), scaler.transform(pca.transform(X_test))

def train_test_kernels(s):
    S_tr = np.array([Statevector(zz_map(s * u)).data for u in U_train])
    S_te = np.array([Statevector(zz_map(s * u)).data for u in U_test])
    return np.abs(S_tr.conj() @ S_tr.T) ** 2, np.abs(S_te.conj() @ S_tr.T) ** 2

kernels = {s: train_test_kernels(s) for s in [np.pi, 0.25]}
for s, (K_tr, K_te) in kernels.items():
    model = SVC(kernel="precomputed", C=10).fit(K_tr, y_train)
    print(f"s = {s:4.2f}: training accuracy {model.score(K_tr, y_train):.3f}, test accuracy {model.score(K_te, y_test):.3f}")

## Your turn: add the shots
Simulate shot noise on both kernels (`rng.binomial(shots, K) / shots`, keeping the training matrix symmetric
with ones on the diagonal) for 100, 1,000 and 10,000 shots, and compare the test accuracies.

In [ ]:
rng = np.random.default_rng(1)

def noisy(K, shots, symmetric):
    E = rng.binomial(shots, np.clip(K, 0, 1)) / shots
    if symmetric:
        E = np.triu(E, 1)
        E = E + E.T
        np.fill_diagonal(E, 1.0)
    return E

for s, (K_tr, K_te) in kernels.items():
    for shots in [100, 1000, 10000]:
        pass     # TODO: fit SVC(kernel="precomputed", C=10) on noisy(K_tr, shots, True); score on noisy(K_te, shots, False)

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
for s, (K_tr, K_te) in kernels.items():
    for shots in [100, 1000, 10000]:
        model = SVC(kernel="precomputed", C=10).fit(noisy(K_tr, shots, True), y_train)
        print(f"s = {s:4.2f}, {shots:6d} shots: test accuracy {model.score(noisy(K_te, shots, False), y_test):.3f}")
print("The concentrated kernel (s = pi) sits near chance with or without shots: it only memorizes.")
print("The small-bandwidth kernel learns, and its accuracy grows with the shot budget. Concentration is")
print("fatal; a small bandwidth fixes it, at the price of enough shots to resolve the differences.")